# Collecte des données

## Sources retenues

| Source | Description | Accès |
|---|---|---|
| LCSQA / Geod'air | Concentrations PM2.5 horaires par station IDF | API (token Geod'air) |
| Météo-France | Variables météo horaires — 8 départements IDF | CSV.gz (data.gouv.fr) |
| IREP / Géorisques | Émissions industrielles PM par installation IDF | ZIP (georisques.gouv.fr) |



## Prérequis

Avant d'exécuter ce notebook, configurer le fichier `.env` 
à partir de `.env.example` avec :
- `GEODAIR_TOKEN` : token API Geod'air (inscription sur geodair.fr)
- `S3_ENDPOINT_URL`, `S3_ACCESS_KEY`, `S3_SECRET_KEY`, `S3_SESSION_TOKEN`, `S3_BUCKET` : credentials SSPCloud

In [1]:
import os
import sys
import boto3
from dotenv import load_dotenv

In [2]:
load_dotenv() # charger les variables d'environnement

# Vérification des variables d'environnement requises
required_vars = [
    "GEODAIR_TOKEN", "S3_ENDPOINT_URL", 
    "S3_ACCESS_KEY", "S3_SECRET_KEY", "S3_BUCKET"
]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Variables manquantes dans .env : {missing}")

print(" Variables d'environnement configurées")

# Client S3
s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("S3_ENDPOINT_URL"),
    aws_access_key_id     = os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key = os.getenv("S3_SECRET_KEY"),
    aws_session_token     = os.getenv("S3_SESSION_TOKEN"),
)
BUCKET = os.getenv("S3_BUCKET")


 Variables d'environnement configurées


## 1. LCSQA / Geod'air — Concentrations PM2.5

**Rôle :** source centrale. Fournit la variable cible (concentration horaire 
de PM2.5) et les features de pollution passée (lags t-1h, t-6h, t-24h).

**Méthode :**
- API Geod'air — polluant PM2.5 (code 39), statistique moyenne horaire (code a1)
- Découpage par trimestre (20 requêtes) pour respecter les limites de l'API
- Filtrage IDF : stations dont le code commence par `FR04`
- Chaque fichier trimestriel est uploadé sur S3 après téléchargement

**Couverture :** 2021 T1  2025 T4

In [3]:
# Ajout du dossier src/ au path pour importer les modules
sys.path.insert(0, os.path.abspath(".."))

from src.collect_lcsqa import collect_lcsqa

# Lance la collecte — skip automatique si les fichiers existent déjà
collect_lcsqa()

# Vérification S3
print("\nFichiers LCSQA sur S3 :")
verifier_s3("projet-qualite-air/raw/lcsqa/")

2026-04-21 04:01:43,845 - INFO - Demarrage collecte LCSQA - 5 trimestres
Trimestres:   0%|          | 0/5 [01:15<?, ?it/s]


KeyboardInterrupt: 

## 2. Météo-France — Données climatologiques horaires

**Rôle :** fournit les features météorologiques explicatives.
La météo est le principal facteur de dispersion ou d'accumulation 
des polluants : le vent disperse, la pluie lessive, les inversions 
thermiques (haute pression, faible vent) accumulent les PM2.5.

**Méthode :**
- API data.gouv.fr — dataset 
- Filtrage : 8 départements IDF (75, 77, 78, 91, 92, 93, 94, 95)
- Format : CSV.gz, téléchargement direct sans authentification
- Variables retenues : température, vent (vitesse + direction), 
  humidité, précipitations

**Couverture :** 2020-2026 — 16 fichiers (2 par département)

In [ ]:
from src.collect_meteo import collect_meteo

# Lance la collecte — skip automatique si les fichiers existent déjà
collect_meteo()

# Vérification S3
print("\nFichiers Météo-France sur S3 :")
verifier_s3("projet-qualite-air/raw/meteo/")

## 3. IREP / Géorisques — Émissions industrielles

**Rôle :** enrichissement spatial statique. Permet de construire une 
variable de densité d'émissions PM autour de chaque station de mesure, 
expliquant pourquoi certaines stations sont structurellement plus polluées.

**Méthode :**
- Téléchargement direct : `https://files.georisques.fr/irep/{annee}.zip`
- Filtrage : départements IDF + milieu AIR uniquement
- Jointure avec `etablissements.csv` pour les coordonnées GPS (WGS84)

**Couverture et limite :**
Les années 2023-2024 présentent uniquement des valeurs `< seuil` 
(inférieures au seuil de déclaration obligatoire) — non exploitables 
quantitativement. La variable IREP est donc disponible uniquement 
pour 2021-2022, ce qui représente 34.6% des observations.
Les NaN pour 2023-2025 sont une limite connue et documentée.

In [ ]:
from src.collect_irep import collect_irep

# Lance la collecte — skip automatique si les fichiers existent déjà
collect_irep()

# Vérification S3
print("\nFichiers IREP sur S3 :")
verifier_s3("projet-qualite-air/raw/irep/")

## Bilan de la collecte

In [ ]:
import pandas as pd

# Bilan des fichiers sur S3
print("=== Bilan complet S3 ===\n")

prefixes = {
    "LCSQA"        : "projet-qualite-air/raw/lcsqa/",
    "Météo-France" : "projet-qualite-air/raw/meteo/",
    "IREP"         : "projet-qualite-air/raw/irep/",
}

bilan = []
for source, prefix in prefixes.items():
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    files    = response.get("Contents", [])
    n_files  = len(files)
    volume   = sum(f["Size"] for f in files) / 1e6
    bilan.append({
        "Source"   : source,
        "Fichiers" : n_files,
        "Volume (Mo)" : round(volume, 1),
        "Statut"   : "bon " if n_files > 0 else "pas bon : ",
    })

pd.DataFrame(bilan).set_index("Source")